## Question 1 - Used cars

**Official data link (Q1):** [Box dataset](https://app.box.com/s/jm6pw202asu4xd3uypwtry2rqk691y1i) — save as **`data/train.csv`** in this `question1` folder.


| Part | Task |
|------|------|
| **(a)** | Find missing values; drop rows with no target **Price**; impute numerics with **median**, **Owner_Type** with **mode** (after parsing numbers in **b**). |
| **(b)** | Strip text units (`kmpl`, `CC`, `bhp`, `Lakh`, etc.) so **Mileage**, **Engine**, **Power**, **New_Price** are plain numbers. |
| **(c)** | One-hot encode **Fuel_Type** and **Transmission**. |
| **(d)** | Add **Car_Age** = current calendar year − **Year**. |
| **(e)** | **pandas**: `select`, `filter`, `rename`, `mutate`, `arrange`, `group_by` + `summarize`. |

**Data:** **`question1/data/train.csv`**; fallback **`Downloads/train.csv`**.  
**Findings write-up (Q1 only):** **`../docs/REPORT.md`**.  
**Target variable:** **`Price`** (lakhs).

In [ ]:
#  Setup
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Project root
_PROJECT_ROOT = Path(r"c:\Users\Rasna\Desktop\UMKC Masters\PDS\Assignemnt-2\question1")

# Dataset.
DATA_PATH = _PROJECT_ROOT / "data" / "train.csv"
if not DATA_PATH.exists():
    DATA_PATH = Path.home() / "Downloads" / "train.csv"

# Used for Car_Age = CURRENT_YEAR - Year 
CURRENT_YEAR = 2026

_OUTPUT_DIR = _PROJECT_ROOT / "outputs"
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading from:", DATA_PATH.resolve())
print("Outputs go to:", _OUTPUT_DIR.resolve())

Loading from: C:\Users\Rasna\Desktop\UMKC Masters\PDS\Assignemnt-2\train.csv
Rows will be saved back under: c:\Users\Rasna\Desktop\UMKC Masters\PDS\Assignemnt-2


### (a) Missing values

- Drop rows where **`Price`** is missing - it is the target; imputing it would invent outcomes.
- After parsing numeric text in (b), impute remaining missing **numeric** columns with the **median** (robust to skew/outliers common in mileage, power, and prices).
- Impute **`Owner_Type`** with the **mode** if any values are missing (most common ownership level).
- **`Name` / `Location`:** if missing, typical practice is mode or `"Unknown"`; this dataset rarely omits them.

In [ ]:
# Load raw CSV 
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

# Some exports include a leading index column with a blank name (e.g. "Unnamed: 0") - drop it.
unnamed = [c for c in df.columns if str(c).lower().startswith("unnamed")]
if unnamed:
    df = df.drop(columns=unnamed)

print("Shape (rows, columns):", df.shape)
print("\nNon-zero missing counts (raw) — use in your report for part (a):")
# display() shows a small table in the notebook; only columns with at least one NaN.
display(df.isna().sum()[lambda s: s > 0])

Shape (rows, columns): (5847, 13)

Non-zero missing counts (raw) — use in your report for part (a):


Mileage         2
Engine         36
Power          36
Seats          38
New_Price    5032
dtype: int64

In [36]:
# Part (a): never impute the target. Price in lakhs is the variable to predict / describe.
n0 = len(df)
df = df[df["Price"].notna()].copy()
print(
    f"Target filter: dropped {n0 - len(df)} row(s) with missing Price. Remaining: {len(df)}"
)

Target filter: dropped 0 row(s) with missing Price. Remaining: 5847


### (b) Strip units-keep numeric values only

Remove suffixes/units such as **kmpl**, **km/kg**, **CC**, **bhp**, and **Lakh** from `Mileage`, `Engine`, `Power`, and `New_Price`.

The helper below keeps the **first number** in each cell (e.g. CNG mileage may show **km/kg**-still one numeric rate).

In [ ]:
def first_number(text: object) -> float:
    """Pull the first number from strings like '19.67 kmpl', '1582 CC', '126.2 bhp', '8.61 Lakh'.

    Non-numeric placeholders (Ask Price, bare null, etc.) become NaN so part (a) can impute later.
    """
    if pd.isna(text):
        return np.nan
    s = str(text).strip()
    # Explicit text placeholders - not convertible to a number.
    if not s or re.match(r"(?i)^(ask|n/?a|null|nan)$", s):
        return np.nan
    # e.g. 'null bhp' with no digits
    if re.search(r"(?i)\bnull\b", s) and not re.search(r"\d", s):
        return np.nan
    # First numeric token (allows comma thousands in theory).
    m = re.search(
        r"([\d]{1,3}(?:,\d{3})*(?:\.\d+)?|\d+(?:\.\d+)?)",
        s.replace(",", ""),
    )
    if not m:
        return np.nan
    try:
        return float(m.group(1))
    except ValueError:
        return np.nan


# Part (b): overwrite these columns with pure floats (units stripped).
for col in ("Mileage", "Engine", "Power", "New_Price"):
    if col in df.columns:
        df[col] = df[col].map(first_number)

# Quick schecks
df[["Mileage", "Engine", "Power", "New_Price", "Price"]].head()

,Mileage,Engine,Power,New_Price,Price
0,19.67,158.0,126.20,NaN,12.50
1,13.00,119.0,88.70,8.61,4.50
2,20.77,124.0,88.76,NaN,6.00
3,15.20,196.0,140.80,NaN,17.74
4,23.08,146.0,63.10,NaN,3.50


#### (a continued) Impute remaining missing values (after numeric parsing)

Fill **numeric** gaps with the column **median** (less sensitive than the mean to extreme mileages, HP, or prices).  
Fill **Owner_Type** with the **mode** when needed. *Rationale:* same as markdown in part (a)

In [38]:
# Columns treated as numeric for median imputation (Price already complete after drop above).
numeric_cols = [
    c
    for c in (
        "Year",
        "Kilometers_Driven",
        "Mileage",
        "Engine",
        "Power",
        "Seats",
        "Price",
        "New_Price",
    )
    if c in df.columns
]

for c in numeric_cols:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].median())

# Categorical ownership label (column name in this file is Owner_Type).
if "Owner_Type" in df.columns and df["Owner_Type"].isna().any():
    mode = df["Owner_Type"].mode(dropna=True)
    df["Owner_Type"] = df["Owner_Type"].fillna(mode.iloc[0] if len(mode) else "Unknown")

# Expect 0 here if every column above was filled; leftover NaNs would be in non-imputed text cols.
print("Total missing values after imputation:", int(df.isna().sum().sum()))

Total missing values after imputation: 0


### (c) One-hot encode `Fuel_Type` and `Transmission`

We use `pd.get_dummies(..., columns=[...])` so new columns are named **`Fuel_Type_*`** and **`Transmission_*`**, and the original two string columns are **dropped**

In [39]:
# Part (c): one-hot; column names match typical get_dummies output (Fuel_Type_Diesel, ...).
df = pd.get_dummies(
    df, columns=["Fuel_Type", "Transmission"], drop_first=False, dtype=int
)

[c for c in df.columns if c.startswith("Fuel_Type_") or c.startswith("Transmission_")]

(['Fuel_Diesel', 'Fuel_Electric', 'Fuel_Petrol'],
 ['Transmission_Automatic', 'Transmission_Manual'])

### (d) New feature - car age

`Car_Age = current_year - Year`

In [40]:
# Part (d): interpretable depreciation / usage proxy (years since manufacture).
df["Car_Age"] = CURRENT_YEAR - df["Year"]
df[["Name", "Year", "Car_Age"]].head()

,Name,Year,Car_Age
0,Hyundai Creta 1.6 CRDi SX Option,2015,11
1,Honda Jazz V,2011,15
2,Maruti Ertiga VDI,2012,14
3,Audi A4 New 2.0 TDI Multitronic,2013,13
4,Nissan Micra Diesel XV,2013,13


### (e) Operations

| Functions | pandas |
|---------------|--------|
| `select()` | column subset `df[list]` |
| `filter()` | boolean mask or `query()` |
| `rename()` | `.rename(columns=...)` |
| `mutate()` | `.assign()` or `df["x"] = ...` |
| `arrange()` | `.sort_values()` |
| `summarize()` + `group_by()` | `.groupby().agg()` |

In [41]:
# Part (e).

# select
cols = [
    "Name",
    "Location",
    "Year",
    "Car_Age",
    "Kilometers_Driven",
    "Price",
    "Power",
] + [
    c
    for c in df.columns
    if c.startswith("Fuel_Type_") or c.startswith("Transmission_")
]
cols = [c for c in cols if c in df.columns]
sel = df[cols].copy()

# filter
flt = sel.query("Price >= 3 and Kilometers_Driven <= 80000")

# rename
flt = flt.rename(columns={"Kilometers_Driven": "Odometer_KM"})

# mutate
flt = flt.assign(
    Price_per_age_year=lambda d: d["Price"] / d["Car_Age"].replace(0, np.nan)
)

# arrange
arr = flt.sort_values(["Location", "Price"], ascending=[True, False])
arr.head(10)

,Name,Location,Year,Car_Age,Odometer_KM,Fuel_Type,Transmission,Price,Power,Price_per_age_year
3471,BMW 3 Series Luxury Line,Ahmedabad,2016,10,27000,Diesel,Automatic,35.00,190.00,3.500000
5712,Mercedes-Benz C-Class Progressive C 220d,Ahmedabad,2019,7,4000,Diesel,Automatic,35.00,194.00,5.000000
2441,BMW 5 Series 2013-2017 520d Luxury Line,Ahmedabad,2016,10,44001,Diesel,Automatic,34.51,190.00,3.451000
1847,Toyota Fortuner 2.8 2WD AT,Ahmedabad,2017,9,15000,Diesel,Automatic,30.00,174.50,3.333333
2599,Mercedes-Benz E-Class 2015-2017 E250 Edition E,Ahmedabad,2016,10,39500,Diesel,Automatic,29.50,204.00,2.950000
1413,Audi Q3 2.0 TDI Quattro,Ahmedabad,2016,10,45000,Diesel,Automatic,27.00,184.00,2.700000
5127,Mercedes-Benz CLA 200 D Sport Edition,Ahmedabad,2015,11,13500,Diesel,Automatic,25.51,136.00,2.319091
3306,Land Rover Freelander 2 HSE SD4,Ahmedabad,2012,14,60000,Diesel,Automatic,23.50,187.74,1.678571
1181,BMW 5 Series 2013-2017 520d Luxury Line,Ahmedabad,2014,12,35000,Diesel,Automatic,23.00,190.00,1.916667
4488,Audi A4 2.0 TDI Multitronic,Ahmedabad,2015,11,13000,Diesel,Automatic,23.00,140.00,2.090909


In [42]:
# summarize + group_by - mean used price, listing count, median car age per city.
grp = (
    df.groupby("Location", as_index=False)
    .agg(
        Avg_Price=("Price", "mean"),
        Count=("Name", "count"),
        Median_Age=("Car_Age", "median"),
    )
    .sort_values("Avg_Price", ascending=False)
)
grp.head(15)

,Location,Avg_Price,Count,Median_Age
3,Coimbatore,15.160206,631,10.0
1,Bangalore,13.482670,352,13.0
7,Kochi,11.309109,640,10.0
5,Hyderabad,9.997423,710,13.0
4,Delhi,9.881944,540,12.0
9,Mumbai,9.592546,762,12.0
0,Ahmedabad,8.567248,218,12.0
2,Chennai,7.958340,476,14.0
10,Pune,6.951000,590,13.0
6,Jaipur,5.916725,403,13.0


### Save processed dataset

In [43]:
# Processed CSV for modeling or submission (same folder layout as first cell).
OUTPUT_PATH = _PROJECT_ROOT / "outputs" / "train_cleaned.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(
    "Saved:",
    OUTPUT_PATH.resolve(),
    f"| rows={len(df)}, cols={df.shape[1]}",
)

Saved: C:\Users\Rasna\Desktop\UMKC Masters\PDS\Assignemnt-2\train_cleaned.csv | rows=5847, cols=19
